# dict の挿入順序に依存する場合は注意する

Python 3.5 以前では、dict をイテレートすると任意の順序でキーが返されるようになっていました。要素を挿入した順序とは無関係な順序でした。

例えば、動物とその子供の呼び名との辞書を作って出力した例を示します。

In [2]:
# Python 3.5 以前の dict の順序は予測できない
baby_names = {
  'cat': 'kitten',
  'dog': 'puppy',
}

print(baby_names)  # Python 3.5 以前では、{'dog': 'puppy', 'cat': 'kitten'} 

{'cat': 'kitten', 'dog': 'puppy'}


辞書を作成したときの順序は、'cat'、'dog' でしたが、出力ではキーの順序が逆の 'dog'、'cat' です。
このような振る舞いには驚かされ、テストケース作成が困難になり、デバッグが難しく、Python 初心者を困惑させます。

これは、それまで辞書型の実装が、組み込み関数 hash と Python 起動時に与えられる乱数のシードとのハッシュ表アルゴリズムに依存していたからです。

Python 3.6、Python 3.7 以降は Python の仕様として、辞書は挿入順を保持するようになりました。

Python 3.5 以前では、イテレーション順序に依存する dict の全メソッドは、key、values、items popitem を含めて、同じようにランダムに見える振る舞いでした。

今では、キーワード引数の順序も、プログラマが関数を呼び出した元の順序になるように保持されています。

しかし、挿入順序の振る舞いが、辞書を扱う場合に常に保持されると仮定すべきではありません。Python では、プログラマが list、dict などに合致する標準プロトコルをエミュレーションするコンテナ型を定義することが簡単にできます（項目43）。

Python は静的型付けではないため、ほとんどのコードは厳格なクラス階層に基づく代わりに、オブジェクトの振る舞いがデファクトの方に基づくダックタイピングに依存しています。これは予期しない結果を招きかねません。

例えば、一番かわいい動物の赤ちゃんというコンテストの結果を示すプログラムを考えます。まず、それぞれの投票数を数えた辞書です。

In [3]:
votes = {
  'otter': 1281,
  'polar bear': 587,
  'fox': 863,
}

この投票データを処理して、動物の順位を空の辞書に登録する関数を定義します。辞書は UI 要素のデータモデルです。

In [5]:
def populate_ranks(votes, ranks):
  names = list(votes.keys())
  names.sort(key=votes.get, reverse=True)
  for i, name in enumerate(names, 1):
    ranks[name] = i

どの動物がコンテストで優勝したかを知らせる関数も必要です。この関数は、populate_ranks が辞書 ranks に昇順で内容を登録しており、先頭のキーが優勝者だと仮定します。

In [6]:
def get_winner(ranks):
  return next(iter(ranks))

これらの関数が設計したとおりに動作して、結果が期待通りになっているか確認します。

In [ ]:
# 期待通り
ranks = {}
populate_ranks(votes, ranks)
print(ranks)  # {'otter': 1, 'fox': 2, 'polar bear': 3}
winner = get_winner(ranks)
print(winner)  # otter

{'otter': 1, 'fox': 2, 'polar bear': 3}
otter


このプログラムの要求が変わったとします。結果を示す UI 要素を順位ではなく、英字順に表示することになりました。そのために、組み込みモジュール collections.abc を使って、新たな辞書的クラスを作り、英字順に内容をイテレートします。

In [ ]:
from collections.abc import MutableMapping

class SortedDict(MutableMapping):
  def __init__(self):
    self.data = {}

  def __getitem__(self, key):
    return self.data[key]

  def __setitem__(self, key, value):
    self.data[key] = value

  def __delitem__(self, key):
    del self.data[key]

  def __iter__(self):
    keys = list(self.data.keys())
    keys.sort() # アルファベット順にソート
    for key in keys:
      yield key

  def __len__(self):
    return len(self.data)

前に定義した関数の標準 dict の代わりに、この SortedDict インスタンスを使いますが、標準辞書のプロトコルに適合しているのでエラーは起こりません。しかし、実行は正しくありません。

In [13]:
sorted_ranks = SortedDict()
populate_ranks(votes, sorted_ranks)
print(sorted_ranks.data)  # {'otter': 1, 'polar bear': 3, 'fox': 2}
winner = get_winner(sorted_ranks)
print(winner)  # otter ではなく fox

{'otter': 1, 'fox': 2, 'polar bear': 3}
fox


get_winner の実装が、辞書のイテレーションで挿入順序が populate_ranks に一致していると仮定しているため、この問題が起こります。このコードは、dict ではなく SortedDict を使っているので、この仮説はもはや成り立ちません。よって、優勝者として返された値は英字順で先頭の 'fox' なのです。

SortedDict の \_\_iter\_\_ 実装がアルファベット順にソートしている。結果 get_winner の iter にて一番最初の要素を返す。つまり、fox が返ってくる。

この問題を解決するには 3つの方法があります。第1は、get_winner 関数を再実装して、ranks 辞書が特定のイテレーション順になっていると仮定しないことです。最も保守的で頑健な解法です。

In [14]:
def get_winner(ranks):
  for name, rank in ranks.items():
    if rank == 1:
      return name

winner = get_winner(sorted_ranks)
print(winner)  # otter

otter


第２の方法は、関数の先頭で ranks の型が期待通りかチェックして、もしそうでないなら例外を送出することです。この解法は、保守的な下位よりも実行性能に優れています。

In [15]:
def get_winner(ranks):
  if not isinstance(ranks, dict):
    raise TypeError('ranks must be a dict')
  return next(iter(ranks))

get_winner(sorted_ranks)  # TypeError: ranks must be a dict

TypeError: ranks must be a dict

第3の解法では、型ヒントを使って get_winner に渡される値が dict インスタンスで辞書的振る舞いをする MutableMapping でないことを確認します。mypy ツールを strict モードで実行することで確認することができます。

In [ ]:
from typing import Dict, MutableMapping

def populate_ranks(votes: Dict[str, int],
                   ranks: Dict[str, int]) -> None:
    names = list(votes.keys())
    names.sort(key=votes.get, reverse=True)
    for i, name in enumerate(names, 1):
        ranks[name] = i

def get_winner(ranks: Dict[str, int]) -> str:
    return next(iter(ranks))

class SortedDict(MutableMapping[str, int]):
    ...

votes = {
  'otter': 1281,
  'polar bear': 587,
  'fox': 863,
}

sorted_ranks = SortedDict()
populate_ranks(votes, sorted_ranks)
print(sorted_ranks.data)  # {'otter': 1, 'polar bear': 3, 'fox': 2}
winner = get_winner(sorted_ranks)  # mypy error
print(winner)  # otter


$ python3 -m mypy --strict example.py で確認できる。

これは正しく dict と MutableMapping 型との間の不整合を検出して、不正な使い方をエラーだと通知します。これは静的型安全性と実行性能との最良の組み合わせです。

## 覚えておくこと

- Python 3.7 以降は、dict インスタンスの内容をイテレーションすると、キーが最初に挿入された順番が保持される。
- Python では、dict インスタンスではないが辞書のように振る舞うオブジェクトを簡単に定義できる。そのような型では、挿入順序が保持されると仮定できない。
- 辞書的クラスで注意するには3通りの方法がある。挿入順序に依存しないコードを書く、実装時に dict 型か明示的にチェックする。あるいは、型ヒントと性的解析を使って dict 値の要件をチェックする